In [ ]:
with open("tiny-shakespeare.txt") as f:
    text = f.read()
print(f"text length: {len(text)} characters")
print(text[:200])

In [ ]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))
print(f"byte length: {len(tokens)}")
print(tokens[:50])

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
top_pair = max(stats, key=stats.get)
print(f"most common pair: {top_pair} -> count {stats[top_pair]}")
print(f"as bytes: {bytes(top_pair)!r}")

In [ ]:
def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

In [ ]:
vocab_size = 512
num_merges = vocab_size - 256

ids = list(tokens)
merges = {}

for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    if (i + 1) % 32 == 0:
        print(f"merge {i+1}/{num_merges}: {pair} -> {idx}")

print(f"\nfinal token sequence length: {len(ids)}")
print(f"compression ratio: {len(tokens) / len(ids):.2f}x")

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

for idx in [256, 270, 300, 400, 500]:
    if idx in vocab:
        print(f"{idx}: {vocab[idx]!r}")

In [ ]:
def decode(ids):
    raw = b"".join(vocab[idx] for idx in ids)
    return raw.decode("utf-8", errors="replace")

print(decode([256, 257, 258]))

In [ ]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

print(encode("hello world"))

In [ ]:
sample = text[:500]
assert decode(encode(sample)) == sample
print("roundtrip OK")
print(f"original chars: {len(sample)}, encoded tokens: {len(encode(sample))}")

In [ ]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        ids = list(text.encode("utf-8"))
        num_merges = vocab_size - 256
        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}
        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = merge(ids, pair, idx)
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose and (i + 1) % 64 == 0:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r})")
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def decode(self, ids):
        raw = b"".join(self.vocab[i] for i in ids)
        return raw.decode("utf-8", errors="replace")

In [ ]:
tok = BPETokenizer()
tok.train(text, vocab_size=512, verbose=True)

sample = "ROMEO: But soft, what light through yonder window breaks?"
ids = tok.encode(sample)
print(f"\nencoded ({len(ids)} tokens): {ids}")
print(f"decoded: {tok.decode(ids)}")
assert tok.decode(tok.encode(sample)) == sample

In [ ]:
import regex as re

GPT2_SPLIT_PATTERN = r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"
gpt2_pat = re.compile(GPT2_SPLIT_PATTERN)

print(gpt2_pat.findall("Hello world! It's 2026."))

In [ ]:
class RegexBPETokenizer(BPETokenizer):
    def __init__(self, pattern=GPT2_SPLIT_PATTERN):
        super().__init__()
        self.pattern = re.compile(pattern)

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        chunks = self.pattern.findall(text)
        ids = [list(ch.encode("utf-8")) for ch in chunks]
        num_merges = vocab_size - 256
        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}
        for i in range(num_merges):
            stats = {}
            for chunk_ids in ids:
                for pair in zip(chunk_ids, chunk_ids[1:]):
                    stats[pair] = stats.get(pair, 0) + 1
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = [merge(c, pair, idx) for c in ids]
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose and (i + 1) % 64 == 0:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]!r})")
        self.merges = merges
        self.vocab = vocab

    def encode(self, text):
        chunks = self.pattern.findall(text)
        out = []
        for ch in chunks:
            ids = list(ch.encode("utf-8"))
            while len(ids) >= 2:
                stats = get_stats(ids)
                pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges:
                    break
                ids = merge(ids, pair, self.merges[pair])
            out.extend(ids)
        return out

In [ ]:
rtok = RegexBPETokenizer()
rtok.train(text, vocab_size=512, verbose=True)

sample = "ROMEO: But soft, what light through yonder window breaks?"
ids = rtok.encode(sample)
print(f"\nencoded ({len(ids)} tokens)")
print(f"decoded: {rtok.decode(ids)}")
assert rtok.decode(rtok.encode(sample)) == sample

for idx in sorted(rtok.vocab)[256:266]:
    print(f"{idx}: {rtok.vocab[idx]!r}")